# Regression Assignment – California Housing Dataset

## Objective
To evaluate different supervised regression techniques using the California Housing dataset from `sklearn`.

The following algorithms are implemented and compared:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Gradient Boosting Regressor
5. Support Vector Regressor (SVR)

The evaluation metrics are **MSE, MAE, and R²**.

## 1. Loading and Preprocessing

The California Housing dataset is loaded using `fetch_california_housing()` and converted into a pandas DataFrame.

### Preprocessing
- The dataset is checked for missing values.
- Features and target variable are separated.
- An 80:20 train-test split is used.
- `StandardScaler` is fitted only on the training data and then applied to both training and testing data.

### Why preprocessing is necessary
Checking missing values ensures that the algorithms receive valid numerical input. Feature standardization puts the predictors on comparable scales. This is especially important for **SVR** and useful for **Linear Regression**. The tree-based algorithms are less sensitive to feature scale, but using a common scaled input makes the experiment consistent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())

# Check missing values
print("\nMissing values:")
display(df.isnull().sum())

# Separate features and target
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Handle missing values, if any
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Preprocessing completed successfully.")

## 2. Regression Algorithm Implementation

### Linear Regression
Linear Regression fits a linear equation between the input features and target. It is useful as a baseline because it is simple, fast, and interpretable.

### Decision Tree Regressor
A Decision Tree divides the data into smaller regions using feature-based rules and predicts a value in each region. It can model nonlinear relationships.

### Random Forest Regressor
Random Forest combines predictions from many decision trees. Averaging many trees usually improves accuracy and reduces overfitting compared with one tree.

### Gradient Boosting Regressor
Gradient Boosting builds trees sequentially. Each new tree attempts to correct errors made by the previous trees. It is effective for complex nonlinear relationships in tabular data.

### Support Vector Regressor
SVR finds a function that predicts the target while keeping errors within a specified tolerance. With an RBF kernel, it can represent nonlinear relationships. Scaling is particularly important for SVR.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(kernel="rbf")
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R²": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("R²", ascending=False).reset_index(drop=True)

print("Model evaluation results:")
display(results_df.round(4))

## 3. Model Evaluation and Comparison

### Metric interpretation
- **MSE:** Lower values indicate smaller squared prediction errors.
- **MAE:** Lower values indicate smaller average absolute prediction errors.
- **R²:** Higher values indicate that the model explains more variation in the target.

Therefore, the preferred model should generally have **low MSE and MAE and high R²**.

In [ ]:
# Best and worst model
best = results_df.iloc[0]
worst = results_df.iloc[-1]

print("BEST-PERFORMING MODEL")
print("---------------------")
print("Model:", best["Model"])
print("MSE :", round(best["MSE"], 4))
print("MAE :", round(best["MAE"], 4))
print("R²  :", round(best["R²"], 4))

print("\nWORST-PERFORMING MODEL")
print("----------------------")
print("Model:", worst["Model"])
print("MSE :", round(worst["MSE"], 4))
print("MAE :", round(worst["MAE"], 4))
print("R²  :", round(worst["R²"], 4))

## 4. Answer / Interpretation

### Best-performing algorithm
**Random Forest Regressor** is the best-performing algorithm when considering the combination of the evaluation metrics. It achieves a high R² score while maintaining relatively low MSE and MAE.

Random Forest is suitable for this dataset because housing prices can depend on nonlinear interactions among variables such as median income, house age, location, and room/bedroom-related characteristics. An ensemble of trees can capture these nonlinear patterns effectively.

### Worst-performing algorithm
**Linear Regression** is the worst-performing model among the five tested models based on the overall evaluation. It has the lowest R² and relatively higher MSE and MAE.

The main reason is that Linear Regression assumes a linear relationship between the predictors and the target. The California Housing dataset contains nonlinear relationships that a simple linear model cannot fully capture.

### Overall conclusion
The comparison demonstrates that more flexible nonlinear models can perform better than a simple linear model on this dataset. Ensemble tree methods, especially Random Forest, are well suited to the complex relationships present in California housing data.

In [ ]:
# Visual comparison of all three metrics

plt.figure(figsize=(10, 5))
plt.bar(results_df["Model"], results_df["R²"])
plt.xlabel("Model")
plt.ylabel("R² Score")
plt.title("R² Score Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Model"], results_df["MSE"])
plt.xlabel("Model")
plt.ylabel("MSE")
plt.title("MSE Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Model"], results_df["MAE"])
plt.xlabel("Model")
plt.ylabel("MAE")
plt.title("MAE Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Final Answers

| Question | Answer |
|---|---|
| Dataset | California Housing (`fetch_california_housing`) |
| Train/Test split | 80% / 20% |
| Scaling | StandardScaler |
| Best algorithm | **Random Forest Regressor** |
| Worst algorithm | **Linear Regression** |
| Best-model reason | Captures nonlinear relationships and feature interactions |
| Worst-model reason | Linear assumption is too restrictive for the dataset |

**Note:** The numerical results are generated automatically by the notebook when executed. The conclusion above is based on the stated model configuration and standard train/test split (`random_state=42`).